In [3]:
import torch
inputs = torch.tensor(
    [[0.43,0.15,0.89],
     [0.55,0.87,0.66],
     [0.57,0.85,0.64],
     [0.22,0.58,0.33],
     [0.77,0.25,0.10],
     [0.05,0.80,0.55]]
)

### Attention 점수 구하기 
쿼리 토큰(임베딩된 입력 토큰 중 하나)와 입력 토큰 사이에 dot product를 통해 어텐션 점수를 구한다. 

In [3]:
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i,x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i,query)
print(attn_scores_2)

#### 계산한 어텐션 정규화하기
어텐션 가중치의 합이 1이 되도록 하기 위함

In [4]:
attn_weight_2_tmp = attn_scores_2 /attn_scores_2.sum()
print("어텐션 가중치:",attn_weight_2_tmp)
print("합:",attn_weight_2_tmp.sum())

In [6]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weight_2_naive = softmax_naive(attn_scores_2)
print("어텐션 가중치:",attn_weight_2_naive)
print("합:",attn_weight_2_naive.sum())

In [9]:
# pytorch에서 제공하는 softmax 함수 사용하기
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("어텐션 가중치:",attn_weights_2)
print("합:",attn_weights_2.sum())

##### 문맥 벡터 계산하기 
임베딩된 입력 토큰과 각 토큰에 해당하는 어텐션 가중치를 곱한 후 모두 더해서 문맥 벡터를 계산

In [11]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

In [ ]:
# 모든 입력 사이의 어텐션 점수 나타내기:for문
attn_scores = torch.empty(6,6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

In [13]:
# 모든 입력 사이의 어텐션 점수 나타내기:행렬 곱
attn_scores = inputs @ inputs.T
print(attn_scores)

In [ ]:
# 정규화하기
attn_weights = torch.softmax(attn_scores, dim=1)
print(attn_weights)

In [15]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

### 훈련 가능한 가중치를 가진 셀프 어텐션 구현하기

In [5]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [6]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in,d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in,d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in,d_out), requires_grad=False)

In [7]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

tensor([0.4306, 1.4551])


In [8]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:",keys.shape)
print("values.shape:",values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [9]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [10]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [11]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5,dim = -1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [13]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


##### 셀프 어텐션 파이썬 클래스 구현 
앞서 진행했던 과정을 하나의 클래스로 구현
- query, key, value 가중치 W 추출  
- q,k,v와 w_q, w_k, w_v 간의 행렬 곱
- query에 대한 key 행렬 곱을 통해 attn_score 얻기
- attn_score을 softmax 함수로 정규화하여 attn_weight로 변환
- 해당 가중치를 values와 곱해서 문맥 벡터 구하기

In [14]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in,d_out))
        self.W_key = nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = nn.Parameter(torch.rand(d_in,d_out))
        
    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
import torch.nn as nn
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)
        
    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec